
## File 04 — End-to-End Pipeline Testing

This notebook tests whether your E-Commerce Lakehouse works across all three medallion layers.




The important distinction: a table existing does not mean the pipeline is correct. We'll test row counts, primary-key uniqueness, required columns, relationships and Gold output availability.


## Testing Architecture:



```text
+----------------------------------+
| 01. CSV LANDING                  |
|                                  |
| Verify source CSV files          |
| Validate file availability       |
| Check source record counts       |
+----------------------------------+
                 |
                 v
+----------------------------------+
| 02. BRONZE DELTA                 |
|                                  |
| Verify Auto Loader ingestion     |
| Validate raw records             |
| Check ingestion metadata         |
+----------------------------------+
                 |
                 v
+----------------------------------+
| 03. SILVER DELTA                 |
|                                  |
| Validate standardized records    |
| Check duplicates and NULL values |
| Verify MERGE results             |
+----------------------------------+
                 |
                 v
+----------------------------------+
| 04. GOLD ANALYTICS               |
|                                  |
| Validate aggregated metrics      |
| Check business KPIs              |
| Verify analytics table counts    |
+----------------------------------+
                 |
                 v
+----------------------------------+
| 05. DATA QUALITY                 |
|                                  |
| Completeness                     |
| Uniqueness                       |
| Validity                         |
| Consistency                      |
+----------------------------------+
                 |
                 v
+----------------------------------+
| 06. REFERENTIAL INTEGRITY        |
|                                  |
| Validate parent-child relations  |
| Identify orphan records          |
| Verify foreign key relationships |
+----------------------------------+
                 |
                 v
+----------------------------------+
| 07. FINAL TEST REPORT            |
|                                  |
| Passed / Failed / Warnings       |
| Record Count Reconciliation      |
| Execution Status                 |
| Final Testing Summary            |
+----------------------------------+
```



## Testing Objectives:



| Stage | Main Testing Objective |
|---|---|
| CSV Landing | Verify source files and input records |
| Bronze Delta | Validate ingestion completeness |
| Silver Delta | Verify cleaning, deduplication, and MERGE |
| Gold Analytics | Validate aggregations and business KPIs |
| Data Quality | Check completeness, uniqueness, validity, and consistency |
| Referential Integrity | Identify invalid parent-child relationships |
| Final Test Report | Consolidate results and determine final status |



## Expected Outcome:


A complete end-to-end testing report that demonstrates whether source data has been correctly ingested, transformed, validated, and aggregated throughout the E-Commerce Lakehouse.


**Important correction:** 

- Data Quality and Referential Integrity are validation checks, not necessarily downstream data layers. 
- They can be executed after Silver processing and before Gold refresh. 
- If Gold is already built, the flow above represents a post-build testing sequence. 
- For production execution, run critical Silver validations before Gold refresh as well.

In [0]:

# ============================================================
# E-COMMERCE LAKEHOUSE
# FOLDER 11: DATABRICKS JOBS AND FINAL TESTING
# FILE 04: END-TO-END PIPELINE TESTING
# ============================================================


# ============================================================
# 1. IMPORTS
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime, timezone
import uuid


# ============================================================
# 2. CONFIGURATION
# ============================================================

CATALOG = "workspace"

BRONZE_SCHEMA = "ecommerce_bronze"
SILVER_SCHEMA = "ecommerce_silver"
GOLD_SCHEMA = "ecommerce_gold"

BASE_PATH = (
    "/Volumes/workspace/default/"
    "data_volume/ecommerce_lakehouse"
)

LANDING_PATH = f"{BASE_PATH}/landing"

TEST_REPORT_TABLE = (
    f"{CATALOG}.{GOLD_SCHEMA}."
    "end_to_end_test_report"
)

TEST_SUMMARY_TABLE = (
    f"{CATALOG}.{GOLD_SCHEMA}."
    "end_to_end_test_summary"
)

DATASETS = [
    "customers",
    "products",
    "orders",
    "order_items",
    "payments",
    "shipments",
    "inventory_snapshots"
]

# Actual Silver table mapping.
# inventory_snapshots is stored as silver_inventory.

SILVER_TABLE_MAP = {
    "customers": "silver_customers",
    "products": "silver_products",
    "orders": "silver_orders",
    "order_items": "silver_order_items",
    "payments": "silver_payments",
    "shipments": "silver_shipments",
    "inventory_snapshots": "silver_inventory"
}

PRIMARY_KEYS = {
    "customers": ["customer_id"],
    "products": ["product_id"],
    "orders": ["order_id"],
    "order_items": ["order_item_id"],
    "payments": ["payment_id"],
    "shipments": ["shipment_id"],
    "inventory_snapshots": [
        "product_id",
        "warehouse_id",
        "snapshot_date"
    ]
}

# Existing Gold tables produced by the pipeline.

GOLD_TABLES = [
    "gold_daily_sales",
    "gold_channel_daily_sales",
    "gold_product_performance",
    "gold_category_performance",
    "gold_customer_lifetime_value",
    "gold_customer_segments",
    "gold_payment_metrics",
    "gold_payment_method_performance",
    "gold_order_payment_summary",
    "gold_shipping_metrics",
    "gold_inventory_status",
    "gold_warehouse_inventory",
    "gold_inventory_snapshot_trends",
    "gold_inventory_alerts",
    "gold_executive_kpis"
]

RUN_ID = str(uuid.uuid4())

TEST_TIMESTAMP = datetime.now(
    timezone.utc
).replace(tzinfo=None)

FAIL_ON_TEST_FAILURE = True

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS "
    f"{CATALOG}.{GOLD_SCHEMA}"
)

print("Configuration loaded successfully.")
print(f"Run ID: {RUN_ID}")


# ============================================================
# 3. TEST REPORT SCHEMA
# ============================================================

test_schema = StructType([

    StructField("test_id", StringType(), False),

    StructField("run_id", StringType(), False),

    StructField("test_category", StringType(), True),

    StructField("test_name", StringType(), True),

    StructField("dataset_name", StringType(), True),

    StructField("table_name", StringType(), True),

    StructField("test_status", StringType(), True),

    StructField("actual_value", StringType(), True),

    StructField("expected_value", StringType(), True),

    StructField("test_message", StringType(), True),

    StructField("tested_at", TimestampType(), True)

])


# ============================================================
# 4. TEST RESULT COLLECTION
# ============================================================

test_results = []


def add_test_result(
    category,
    test_name,
    dataset_name,
    table_name,
    passed,
    actual=None,
    expected=None,
    message=None
):

    test_results.append({

        "test_id": str(uuid.uuid4()),

        "run_id": RUN_ID,

        "test_category": category,

        "test_name": test_name,

        "dataset_name": dataset_name,

        "table_name": table_name,

        "test_status": (
            "PASSED" if passed else "FAILED"
        ),

        "actual_value": (
            None if actual is None else str(actual)
        ),

        "expected_value": (
            None if expected is None else str(expected)
        ),

        "test_message": message,

        "tested_at": TEST_TIMESTAMP

    })


# ============================================================
# 5. TABLE NAME HELPERS
# ============================================================

def bronze_table(dataset):

    return (
        f"{CATALOG}.{BRONZE_SCHEMA}.{dataset}"
    )


def silver_table(dataset):

    silver_name = SILVER_TABLE_MAP[dataset]

    return (
        f"{CATALOG}.{SILVER_SCHEMA}.{silver_name}"
    )


def gold_table(table):

    return (
        f"{CATALOG}.{GOLD_SCHEMA}.{table}"
    )


def table_exists(table_name):

    return spark.catalog.tableExists(
        table_name
    )


def read_table_if_exists(table_name):

    if table_exists(table_name):

        return spark.table(table_name)

    return None


def check_required_columns(
    df,
    required_columns
):

    return [
        column
        for column in required_columns
        if column not in df.columns
    ]


# ============================================================
# 6. TEST LANDING CSV FILES
# ============================================================

print("\nTEST 1: LANDING FILE VALIDATION")


for dataset in DATASETS:

    file_path = (
        f"{LANDING_PATH}/{dataset}/{dataset}.csv"
    )

    parent_path = (
        f"{LANDING_PATH}/{dataset}"
    )

    try:

        files = dbutils.fs.ls(parent_path)

        matching_files = [

            file.name for file in files

            if file.name == f"{dataset}.csv"

        ]

        exists = len(matching_files) > 0

        add_test_result(

            "LANDING",

            "CSV file existence",

            dataset,

            file_path,

            exists,

            actual=exists,

            expected=True,

            message=(
                "Landing CSV found."
                if exists
                else "Landing CSV is missing."
            )

        )

    except Exception as error:

        add_test_result(

            "LANDING",

            "CSV file existence",

            dataset,

            file_path,

            False,

            actual="Not accessible",

            expected=True,

            message=str(error)[:1000]

        )


# ============================================================
# 7. TEST BRONZE TABLE EXISTENCE AND ROW COUNTS
# ============================================================

print("\nTEST 2: BRONZE VALIDATION")


bronze_counts = {}


for dataset in DATASETS:

    table_name = bronze_table(dataset)

    exists = table_exists(table_name)

    add_test_result(

        "BRONZE",

        "Bronze table existence",

        dataset,

        table_name,

        exists,

        actual=exists,

        expected=True

    )

    if not exists:

        continue

    try:

        df = spark.table(table_name)

        row_count = df.count()

        bronze_counts[dataset] = row_count

        add_test_result(

            "BRONZE",

            "Bronze non-empty check",

            dataset,

            table_name,

            row_count > 0,

            actual=row_count,

            expected="Greater than 0"

        )

    except Exception as error:

        add_test_result(

            "BRONZE",

            "Bronze row count readable",

            dataset,

            table_name,

            False,

            actual=str(error)[:500],

            expected="Readable Delta table"

        )


# ============================================================
# 8. TEST SILVER TABLE EXISTENCE AND ROW COUNTS
# ============================================================

print("\nTEST 3: SILVER VALIDATION")


silver_counts = {}


for dataset in DATASETS:

    table_name = silver_table(dataset)

    exists = table_exists(table_name)

    add_test_result(

        "SILVER",

        "Silver table existence",

        dataset,

        table_name,

        exists,

        actual=exists,

        expected=True

    )

    if not exists:

        continue

    try:

        df = spark.table(table_name)

        row_count = df.count()

        silver_counts[dataset] = row_count

        add_test_result(

            "SILVER",

            "Silver non-empty check",

            dataset,

            table_name,

            row_count > 0,

            actual=row_count,

            expected="Greater than 0"

        )

    except Exception as error:

        add_test_result(

            "SILVER",

            "Silver row count readable",

            dataset,

            table_name,

            False,

            actual=str(error)[:500],

            expected="Readable Delta table"

        )


# ============================================================
# 9. TEST PRIMARY KEY NULLS AND DUPLICATES
# ============================================================

print("\nTEST 4: PRIMARY KEY VALIDATION")


for dataset in DATASETS:

    table_name = silver_table(dataset)

    if not table_exists(table_name):

        continue

    df = spark.table(table_name)

    keys = PRIMARY_KEYS[dataset]

    missing_keys = check_required_columns(
        df,
        keys
    )

    if missing_keys:

        add_test_result(

            "PRIMARY_KEY",

            "Primary key columns exist",

            dataset,

            table_name,

            False,

            actual=missing_keys,

            expected=keys,

            message="Required primary key columns are missing."

        )

        continue

    null_condition = None

    for key in keys:

        condition = F.col(key).isNull()

        null_condition = (
            condition
            if null_condition is None
            else null_condition | condition
        )

    null_key_count = (

        df.filter(null_condition)

        .count()

    )

    add_test_result(

        "PRIMARY_KEY",

        "Null primary key check",

        dataset,

        table_name,

        null_key_count == 0,

        actual=null_key_count,

        expected=0

    )

    duplicate_count = (

        df.groupBy(*keys)

        .count()

        .filter(F.col("count") > 1)

        .count()

    )

    add_test_result(

        "PRIMARY_KEY",

        "Duplicate primary key check",

        dataset,

        table_name,

        duplicate_count == 0,

        actual=duplicate_count,

        expected=0

    )


# ============================================================
# 10. TEST REFERENTIAL INTEGRITY
# ============================================================

print("\nTEST 5: REFERENTIAL INTEGRITY")


RELATIONSHIPS = [

    (
        "orders",
        "customer_id",
        "customers",
        "customer_id"
    ),

    (
        "order_items",
        "order_id",
        "orders",
        "order_id"
    ),

    (
        "order_items",
        "product_id",
        "products",
        "product_id"
    ),

    (
        "payments",
        "order_id",
        "orders",
        "order_id"
    ),

    (
        "shipments",
        "order_id",
        "orders",
        "order_id"
    ),

    (
        "inventory_snapshots",
        "product_id",
        "products",
        "product_id"
    )

]


for (
    child_dataset,
    child_key,
    parent_dataset,
    parent_key
) in RELATIONSHIPS:

    child_table = silver_table(
        child_dataset
    )

    parent_table = silver_table(
        parent_dataset
    )

    relationship_name = (
        f"{child_dataset}.{child_key} to "
        f"{parent_dataset}.{parent_key}"
    )

    if not (
        table_exists(child_table)
        and table_exists(parent_table)
    ):

        add_test_result(

            "REFERENTIAL_INTEGRITY",

            relationship_name,

            child_dataset,

            child_table,

            False,

            actual="Required table missing",

            expected="Both tables available"

        )

        continue

    child_df = spark.table(
        child_table
    )

    parent_df = spark.table(
        parent_table
    )

    missing_child_columns = check_required_columns(
        child_df,
        [child_key]
    )

    missing_parent_columns = check_required_columns(
        parent_df,
        [parent_key]
    )

    if missing_child_columns or missing_parent_columns:

        add_test_result(

            "REFERENTIAL_INTEGRITY",

            relationship_name,

            child_dataset,

            child_table,

            False,

            actual={
                "missing_child_columns": missing_child_columns,
                "missing_parent_columns": missing_parent_columns
            },

            expected="Required foreign key columns available"

        )

        continue

    child_keys = child_df.select(

        F.col(child_key).alias("_fk")

    )

    parent_keys = (

        parent_df

        .select(
            F.col(parent_key).alias("_pk")
        )

        .filter(
            F.col("_pk").isNotNull()
        )

        .distinct()

    )

    orphan_count = (

        child_keys

        .filter(
            F.col("_fk").isNotNull()
        )

        .join(

            parent_keys,

            F.col("_fk") == F.col("_pk"),

            "left_anti"

        )

        .count()

    )

    add_test_result(

        "REFERENTIAL_INTEGRITY",

        relationship_name,

        child_dataset,

        child_table,

        orphan_count == 0,

        actual=orphan_count,

        expected=0,

        message=(
            "Orphan references found."
            if orphan_count > 0
            else "No orphan references found."
        )

    )


# ============================================================
# 11. TEST GOLD TABLE AVAILABILITY
# ============================================================

print("\nTEST 6: GOLD VALIDATION")


gold_counts = {}


for table in GOLD_TABLES:

    table_name = gold_table(table)

    exists = table_exists(table_name)

    add_test_result(

        "GOLD",

        "Gold table existence",

        table,

        table_name,

        exists,

        actual=exists,

        expected=True

    )

    if not exists:

        continue

    try:

        count = spark.table(
            table_name
        ).count()

        gold_counts[table] = count

        add_test_result(

            "GOLD",

            "Gold table non-empty check",

            table,

            table_name,

            count > 0,

            actual=count,

            expected="Greater than 0"

        )

    except Exception as error:

        add_test_result(

            "GOLD",

            "Gold table readable",

            table,

            table_name,

            False,

            actual=str(error)[:500],

            expected="Readable Delta table"

        )


# ============================================================
# 12. TEST DATA QUALITY REPORT AVAILABILITY
# ============================================================

print("\nTEST 7: DATA QUALITY REPORTS")


QUALITY_TABLES = [

    "data_quality_report",

    "completeness_uniqueness_report",

    "data_type_format_report",

    "business_rule_validation_report",

    "quarantine_records",

    "data_quality_dashboard_summary"

]


for table in QUALITY_TABLES:

    table_name = (
        f"{CATALOG}.{SILVER_SCHEMA}.{table}"
    )

    exists = table_exists(table_name)

    add_test_result(

        "DATA_QUALITY",

        "Quality report availability",

        table,

        table_name,

        exists,

        actual=exists,

        expected=True

    )


# ============================================================
# 13. TEST REFERENTIAL INTEGRITY REPORTS
# ============================================================

print("\nTEST 8: REFERENTIAL INTEGRITY REPORTS")


RI_TABLES = [

    "referential_integrity_report",

    "referential_integrity_orphans",

    "orphan_record_quarantine",

    "foreign_key_relationship_report",

    "foreign_key_cardinality_report",

    "referential_integrity_dashboard_summary"

]


for table in RI_TABLES:

    table_name = (
        f"{CATALOG}.{SILVER_SCHEMA}.{table}"
    )

    exists = table_exists(table_name)

    add_test_result(

        "REFERENTIAL_INTEGRITY_REPORT",

        "Integrity report availability",

        table,

        table_name,

        exists,

        actual=exists,

        expected=True

    )


# ============================================================
# 14. TEST BRONZE AND SILVER ROW COUNT COMPARISON
# ============================================================

print("\nTEST 9: BRONZE AND SILVER COUNT COMPARISON")


for dataset in DATASETS:

    bronze_count = bronze_counts.get(
        dataset
    )

    silver_count = silver_counts.get(
        dataset
    )

    if (
        bronze_count is None
        or silver_count is None
    ):

        add_test_result(

            "ROW_COUNT_COMPARISON",

            "Bronze versus Silver row count",

            dataset,

            silver_table(dataset),

            False,

            actual={
                "bronze": bronze_count,
                "silver": silver_count
            },

            expected="Both row counts available",

            message="Unable to compare because a table count is missing."

        )

        continue

    # Silver can legitimately contain fewer records
    # because invalid records are quarantined.
    #
    # This is a basic sanity check, not proof
    # of complete or correct processing.

    add_test_result(

        "ROW_COUNT_COMPARISON",

        "Bronze versus Silver row count",

        dataset,

        silver_table(dataset),

        silver_count <= bronze_count,

        actual={
            "bronze": bronze_count,
            "silver": silver_count
        },

        expected=(
            "Silver count less than or equal "
            "to Bronze count"
        ),

        message=(
            "Basic row-count sanity check. "
            "Does not prove complete processing."
        )

    )


# ============================================================
# 15. CREATE TEST REPORT DATAFRAME
# ============================================================

if not test_results:

    raise RuntimeError(
        "No test results were generated."
    )


test_report_df = spark.createDataFrame(

    test_results,

    schema=test_schema

)


# ============================================================
# 16. SAVE TEST REPORT
# ============================================================

(

    test_report_df

    .write

    .format("delta")

    .mode("append")

    .saveAsTable(TEST_REPORT_TABLE)

)

print(
    f"Test report saved: {TEST_REPORT_TABLE}"
)


# ============================================================
# 17. BUILD TEST SUMMARY
# ============================================================

test_summary_df = (

    test_report_df

    .groupBy(
        "run_id"
    )

    .agg(

        F.count("*").alias(
            "total_tests"
        ),

        F.sum(

            F.when(
                F.col("test_status") == "PASSED",
                1
            ).otherwise(0)

        ).alias("passed_tests"),

        F.sum(

            F.when(
                F.col("test_status") == "FAILED",
                1
            ).otherwise(0)

        ).alias("failed_tests")

    )

    .withColumn(

        "pass_rate",

        F.round(

            F.col("passed_tests")
            / F.col("total_tests") * 100,

            2

        )

    )

    .withColumn(

        "overall_status",

        F.when(
            F.col("failed_tests") == 0,
            "PASSED"
        ).otherwise("FAILED")

    )

    .withColumn(

        "tested_at",

        F.lit(TEST_TIMESTAMP).cast("timestamp")

    )

)


# ============================================================
# 18. SAVE TEST SUMMARY
# ============================================================

(

    test_summary_df

    .write

    .format("delta")

    .mode("append")

    .saveAsTable(TEST_SUMMARY_TABLE)

)

print(
    f"Test summary saved: {TEST_SUMMARY_TABLE}"
)


# ============================================================
# 19. DISPLAY TEST RESULTS
# ============================================================

print("\nEND-TO-END TEST RESULTS")

display(

    test_report_df.orderBy(

        "test_category",

        "dataset_name",

        "test_name"

    )

)


# ============================================================
# 20. DISPLAY TEST SUMMARY
# ============================================================

print("\nEND-TO-END TEST SUMMARY")

display(
    test_summary_df
)


# ============================================================
# 21. DISPLAY FAILED TESTS
# ============================================================

failed_tests_df = (

    test_report_df

    .filter(
        F.col("test_status") == "FAILED"
    )

)

print("\nFAILED TESTS")

display(
    failed_tests_df
)


# ============================================================
# 22. FINAL TEST STATUS
# ============================================================

summary = test_summary_df.first()

total_tests = summary["total_tests"]

passed_tests = summary["passed_tests"]

failed_tests = summary["failed_tests"]

pass_rate = summary["pass_rate"]

overall_status = summary["overall_status"]


print("\n" + "=" * 70)

print("END-TO-END PIPELINE TEST SUMMARY")

print("=" * 70)

print(f"Run ID: {RUN_ID}")

print(f"Total tests: {total_tests}")

print(f"Passed tests: {passed_tests}")

print(f"Failed tests: {failed_tests}")

print(f"Pass rate: {pass_rate}%")

print(f"Status: {overall_status}")

print("=" * 70)


# ============================================================
# 23. FINAL FAILURE HANDLING
# ============================================================

if FAIL_ON_TEST_FAILURE and failed_tests > 0:

    raise RuntimeError(

        f"End-to-end testing failed: "
        f"{failed_tests} test(s) failed. "
        f"Review {TEST_REPORT_TABLE} "
        f"for Run ID {RUN_ID}."

    )


print(
    "END-TO-END TESTING COMPLETED SUCCESSFULLY."
)


In [0]:
%sql
SELECT
    test_category,
    test_name,
    dataset_name,
    table_name,
    actual_value,
    expected_value,
    test_message,
    tested_at
FROM workspace.ecommerce_gold.end_to_end_test_report
WHERE run_id = 'de5e4986-179d-4e72-a86f-c0ea3c9f3414'
  AND test_status = 'FAILED'
ORDER BY test_category, dataset_name;

In [0]:
%sql
SELECT *
FROM workspace.ecommerce_gold.end_to_end_test_summary
WHERE run_id = 'de5e4986-179d-4e72-a86f-c0ea3c9f3414';

In [0]:
%sql
SHOW TABLES IN workspace.ecommerce_silver;

In [0]:
%sql
SHOW TABLES IN workspace.ecommerce_gold;

In [0]:
%sql
DESCRIBE TABLE workspace.ecommerce_silver.silver_inventory;

SELECT COUNT(*) AS silver_inventory_count
FROM workspace.ecommerce_silver.silver_inventory;

SELECT *
FROM workspace.ecommerce_silver.silver_inventory
LIMIT 5;

In [0]:
%sql
DESCRIBE TABLE workspace.ecommerce_gold.gold_order_payment_summary;

SELECT COUNT(*) AS gold_order_payment_count
FROM workspace.ecommerce_gold.gold_order_payment_summary;

SELECT *
FROM workspace.ecommerce_gold.gold_order_payment_summary
LIMIT 5;


## SQL — Inspect the test results


### Overall test summary

In [0]:
%sql

SELECT *
FROM workspace.ecommerce_gold.end_to_end_test_summary
ORDER BY tested_at DESC;


### Find failed tests

In [0]:
%sql

SELECT
    test_category,
    dataset_name,
    test_name,
    actual_value,
    expected_value,
    test_message
FROM workspace.ecommerce_gold.end_to_end_test_report
WHERE test_status = 'FAILED'
ORDER BY tested_at DESC;


### Dataset-level test results

In [0]:
%sql

SELECT
    dataset_name,
    test_category,
    COUNT(*) AS total_tests,
    SUM(
        CASE WHEN test_status = 'PASSED'
        THEN 1 ELSE 0 END
    ) AS passed_tests,
    SUM(
        CASE WHEN test_status = 'FAILED'
        THEN 1 ELSE 0 END
    ) AS failed_tests
FROM workspace.ecommerce_gold.end_to_end_test_report
GROUP BY dataset_name, test_category
ORDER BY dataset_name, test_category;


### Latest test run

In [0]:
%sql

SELECT *
FROM workspace.ecommerce_gold.end_to_end_test_report
WHERE run_id = (
    SELECT run_id
    FROM workspace.ecommerce_gold.end_to_end_test_summary
    ORDER BY tested_at DESC
    LIMIT 1
)
ORDER BY test_category, dataset_name;